# Paged Attention — Colab runner

Mounts Google Drive, installs deps, then drives the test scripts via `!python -m ...`. Project is expected at `/content/drive/MyDrive/PagedAttention` — edit `PROJECT_DIR` below if yours lives elsewhere.

Pick a GPU runtime: **Runtime → Change runtime type → T4 / A100**. Phase 3 will fall back to a CPU torch reference if no GPU is available.

## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. `cd` into the project

In [2]:
import os
PROJECT_DIR = '/content/drive/MyDrive/PagedAttention'
os.chdir(PROJECT_DIR)
!pwd && ls

/content/drive/MyDrive/PagedAttention
bench		 continuous_cache.py  doc  paged_cache.py  __pycache__
colab_run.ipynb  cpu		      gpu  plots	   serving


## 3. Install deps

`torch` ships with the Colab image. `transformers` and `triton` are added explicitly — `triton` is what lets the Phase 3 paged-decode kernel use the GPU backend.

In [3]:
!pip install -q "transformers>=5.1.0" triton numpy datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 88.6 MB/s eta 0:00:00


## 4. Sanity check — GPU + library versions

In [4]:
import torch
print('torch', torch.__version__)
print('cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device', torch.cuda.get_device_name(0))
try:
    import triton
    print('triton', triton.__version__)
except ImportError:
    print('triton: NOT INSTALLED')

torch 2.10.0+cu128
cuda True
device Tesla T4
triton 3.6.0


## 5. Synthetic kernel correctness

Builds random paged KV with variable per-sequence lengths, compares the kernel (torch reference + Triton if CUDA) against naive `softmax(QK^T/√d)V`. Expected: fp32/CPU < `1e-5`, fp16/Triton/CUDA < `5e-3`.

In [5]:
!python -m gpu.test_paged_kernel


--- fp32 / cpu  device=cpu  dtype=torch.float32 ---
  paged-torch vs naive   max|diff| = 2.38e-07  (tol 1e-05)
  paged-triton: skipped (triton=True, device=cpu)

--- fp16 / cuda  device=cuda  dtype=torch.float16 ---
  paged-torch vs naive   max|diff| = 0.00e+00  (tol 5e-03)
  paged-triton vs naive  max|diff| = 9.77e-04  (tol 5e-03)

=== PASS ===


## 6. Headline experiments — E1 / E3 / E5 / E6

- **E1** PagedCache vs DynamicCache correctness (B=1, greedy)
- **E3** Prefix sharing, N=8 parallel completions of a 256-token prompt
- **E5** Continuous (v2) vs uniform (v1) batching on a mixed-length workload
- **E6** Paged-decode kernel parity — re-runs E1 with the Triton kernel routed in via the monkey-patch

The first paged-decode forward pass may pause briefly while Triton autotunes `num_warps` × `num_stages` for `(D=64, BLOCK_SIZE=16)`. The winning config is cached for the rest of the session.

In [6]:
!mkdir -p results && python -m gpu.run_all 2>&1 | tee results/run_all_$(date +%Y%m%d_%H%M%S).log

device=cuda dtype=torch.float16

loading gpt2...
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 672.50it/s] 
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.

=== E1 — correctness vs DynamicCache (B=1, greedy) ===
  [PASS] 'The capital of France is'
  [PASS] 'Once upon a time'
  [PASS] 'Roses are red,'
  [PASS] 'The quick brown fox jumps over the'
  4/4 match  blocks_remaining=0

=== E3 — prefix sharing (N=8 parallel of one 256-tok prompt) ===
  indep   pre=1536  post=1632
  shared  pre=192  post=288
  prefill ratio: 8.00x  (theoretical max = N = 8)
  total ratio:   5.67x

=== E5 — continuous (v2) vs uniform (v1) ===
  workload N=16, source=no_robots, prompt_lens p50/p95/max=27/173/206, output_lens p50/p95/max=185/256/256
  v1 wall 13.29s  (182.1 tok/s)
  v2 wall 11.40s  (212.3 tok/s)


## 7. (Optional) Individual CPU experiments

These are device-pinned to CPU and contain per-experiment plotting code. Skip unless you want the plots saved to `cpu/experiments/`. Uncomment the ones you want.

In [7]:
# !python -m cpu.experiments.e1_correctness
# !python -m cpu.experiments.e2_fragmentation
# !python -m cpu.experiments.e3_prefix_sharing
# !python -m cpu.experiments.e4_throughput
# !python -m cpu.experiments.e5_continuous_vs_uniform